In [ ]:
# Debug: test healpy.boundaries shape
import healpy as hp
import numpy as np

nside_test = 32
pixels_test = np.arange(12 * nside_test**2, dtype=int)[:5]  # First 5 pixels
print(f"Testing with nside={nside_test}, pixels shape: {pixels_test.shape}")
print(f"Pixels: {pixels_test}")

xyz = hp.boundaries(nside_test, pixels_test, step=1, nest=True)
print(f"\nxyz shape from hp.boundaries: {xyz.shape}")
print(f"Expected: (3, 4, {len(pixels_test)})")

x, y, z = xyz[0], xyz[1], xyz[2]
print(f"\nx shape: {x.shape}")
print(f"y shape: {y.shape}")  
print(f"z shape: {z.shape}")

Testing with nside=32, pixels shape: (5,)
Pixels: [0 1 2 3 4]

xyz shape from hp.boundaries: (5, 3, 4)
Expected: (3, 4, 5)

x shape: (3, 4)
y shape: (3, 4)
z shape: (3, 4)


# Geospatial

> HEALPix → vector utilities: produce polygon geometries for HEALPix cells and save as GeoParquet

In [ ]:
#| default_exp geospatial

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Iterable, Optional, Tuple, List, Union
from pathlib import Path
import math
import numpy as np
import pandas as pd
import healpy as hp
from shapely.geometry import Polygon, mapping
from shapely import wkb
import geopandas as gpd
import pyarrow as pa
import pyarrow.parquet as pq
import json
import antimeridian
import warnings
from tqdm.auto import tqdm

## Core helpers

In [ ]:
#| export
def _healpy_boundaries_lonlat(nside: int, pixels: np.ndarray, nest: bool = True) -> Tuple[np.ndarray, np.ndarray]:
    """Return corner longitudes and latitudes for given pixels.
    Uses vectorized `healpy.boundaries`.
    Returns shapes: (npix, ncorner) for lons and lats in degrees.
    """
    # healpy.boundaries with array input returns shape (npix, 3, ncorners) for Cartesian (x,y,z)
    xyz = hp.boundaries(nside, pixels, step=1, nest=nest)  # shape (npix, 3, 4)
    
    # Extract x, y, z components: shape (npix, 4) each
    x = xyz[:, 0, :]  # shape (npix, 4)
    y = xyz[:, 1, :]  # shape (npix, 4)
    z = xyz[:, 2, :]  # shape (npix, 4)
    
    # Convert Cartesian to spherical (theta, phi)
    theta = np.arccos(z)  # polar angle in radians, shape (npix, 4)
    phi = np.arctan2(y, x)  # azimuthal angle in radians, shape (npix, 4)
    
    # Convert to degrees and to lon/lat
    lats = 90.0 - np.degrees(theta)  # shape (npix, 4)
    lons = np.degrees(phi)  # shape (npix, 4), in [-180, 180]
    lons = np.mod(lons, 360.0)  # normalize to [0, 360)
    
    # Already in correct shape: (npix, 4)
    return lons, lats

In [ ]:
#| export
def _normalize_lon(lons: np.ndarray, convention: str = '0_360') -> np.ndarray:
    """Normalize longitude array to given convention.
    convention: '0_360' or '-180_180'
    lons: array-like of longitudes in degrees
    Returns same-shaped array.
    """
    lons = np.asarray(lons, dtype=float)
    if convention == '0_360':
        lons = np.mod(lons, 360.0)
    else:
        # map to (-180, 180]
        lons = ((lons + 180.0) % 360.0) - 180.0
    return lons

## Polygon creation and antimeridian handling

In [ ]:
#| export
def _make_polygon_from_corners(lons: List[float], lats: List[float], lon_convention: str = '0_360', fix_antimeridian: bool = True) -> Polygon:
    """Make a shapely Polygon from corner lon/lat lists.
    Automatically fixes antimeridian-wrapping using `antimeridian.fix_polygon` when requested.
    """
    # Normalize input longitudes to requested convention
    lons = np.asarray(lons, dtype=float)
    lats = np.asarray(lats, dtype=float)
    lons = _normalize_lon(lons, convention=lon_convention)
    # Build polygon coords in lon,lat order
    coords = list(zip(lons.tolist(), lats.tolist()))
    poly = Polygon(coords)
    if fix_antimeridian:
        try:
            fixed = antimeridian.fix_polygon(poly)
            return fixed
        except Exception:
            # Fallback: return original polygon but warn
            warnings.warn('antimeridian.fix_polygon failed; returning raw polygon')
            return poly
    return poly

## Main API: build GeoDataFrame and save geoparquet

In [ ]:
#| export
def healpix_to_geodataframe(nside: int, order: str = 'nested', lon_convention: str = '0_360',
                              pixels: Optional[Iterable[int]] = None, fix_antimeridian: bool = True,
                              chunk_size: int = 65536) -> gpd.GeoDataFrame:
    """Create a GeoDataFrame of HEALPix cell polygons.
    
    Args:
        nside: HEALPix nside
        order: 'nested' or 'ring'
        lon_convention: '0_360' or '-180_180' (affects polygon coordinates)
        pixels: optional iterable of pixel indices; default = all pixels
        fix_antimeridian: whether to call `antimeridian.fix_polygon` on polygons crossing the meridian
        chunk_size: number of pixels to process per chunk for memory control
    
    Returns:
        GeoDataFrame with columns: 'healpix_id' and 'geometry' (EPSG:4326)
    """
    nest = True if order == 'nested' else False
    npix = hp.nside2npix(nside)
    if pixels is None:
        pixels = np.arange(npix, dtype=int)
    else:
        pixels = np.asarray(list(pixels), dtype=int)
    
    # Process in chunks to control memory and allow large nside
    records = []
    total = len(pixels)
    
    with tqdm(total=total, desc=f"Building HEALPix geometries (nside={nside})", unit="cell") as pbar:
        for start in range(0, total, chunk_size):
            end = min(start + chunk_size, total)
            pix_chunk = pixels[start:end]
            lons_arr, lats_arr = _healpy_boundaries_lonlat(nside, pix_chunk, nest=nest)
            # lons_arr, lats_arr shapes: (len(pix_chunk), ncorner)
            for i, pix in enumerate(pix_chunk):
                poly = _make_polygon_from_corners(lons_arr[i], lats_arr[i], lon_convention=lon_convention, fix_antimeridian=fix_antimeridian)
                records.append({'healpix_id': int(pix), 'geometry': poly})
            
            pbar.update(len(pix_chunk))
    
    gdf = gpd.GeoDataFrame(records, geometry='geometry', crs='EPSG:4326')
    gdf = gdf.set_index('healpix_id')
    return gdf

In [ ]:
#| export
def _load_metadata_for_aggregate(agg_path: Path) -> Optional[dict]:
    """Load metadata JSON sidecar for an aggregate parquet file.
    
    Looks for {agg_stem}.meta.json in the same directory.
    Returns dict if found, None otherwise (no error).
    """
    meta_path = agg_path.parent / f'{agg_path.stem}.meta.json'
    if meta_path.exists():
        try:
            with open(meta_path, 'r') as f:
                return json.load(f)
        except Exception as e:
            warnings.warn(f'Failed to load metadata {meta_path}: {e}')
            return None
    return None


#| export
def _extract_healpix_params_from_metadata(metadata: dict) -> dict:
    """Extract nside, order, lon_convention from metadata.
    
    Returns dict with keys (may be empty if not found):
        - 'nside': int or None
        - 'order': 'nested'/'ring' or None
        - 'lon_convention': '0_360'/'-180_180' or None
    """
    result = {'nside': None, 'order': None, 'lon_convention': None}
    try:
        # Look in sidecar_metadata.healpix
        healpix_meta = metadata.get('sidecar_metadata', {}).get('healpix', {})
        if healpix_meta:
            result['nside'] = healpix_meta.get('nside')
            order_str = healpix_meta.get('order', '').lower()
            if order_str in ('nested', 'ring'):
                result['order'] = order_str
        
        # Look in sidecar_metadata.coordinates
        coords_meta = metadata.get('sidecar_metadata', {}).get('coordinates', {})
        if coords_meta:
            lon_conv = coords_meta.get('lon_convention')
            if lon_conv in ('0_360', '-180_180'):
                result['lon_convention'] = lon_conv
    except Exception as e:
        warnings.warn(f'Error extracting HEALPix params from metadata: {e}')
    
    return result


#| export
def _validate_aggregate_file(metadata: dict, input_path: Path) -> None:
    """Validate that the input file is an aggregate, not a sidecar.
    
    Raises ClickException if file is wrong type.
    """
    import click
    if metadata:
        stage = metadata.get('processing', {}).get('stage')
        if stage == 'sidecar':
            raise click.ClickException(
                f'\n❌ ERROR: Input file is a SIDECAR, not an AGGREGATE!\n\n'
                f'   File: {input_path.name}\n'
                f'   Stage: {stage}\n\n'
                f'You must pass the aggregate output from healpyxel_aggregate, not the sidecar.\n'
                f'Look for a file named: *aggregate*.parquet\n'
            )
        elif stage not in ('aggregate', None):
            raise click.ClickException(
                f'\n❌ ERROR: Unexpected processing stage: {stage}\n\n'
                f'   File: {input_path.name}\n\n'
                f'Expected: aggregate (output from healpyxel_aggregate)\n'
                f'For more info, check the .meta.json sidecar: {input_path.stem}.meta.json\n'
            )


In [ ]:
#| export
def save_healpix_to_geoparquet(nside: int, output_path: Union[str, Path], order: str = 'nested',
                               lon_convention: str = '0_360', fix_antimeridian: bool = True,
                               chunk_size: int = 65536, parquet_kwargs: Optional[dict] = None) -> Path:
    """Build HEALPix vector layer and save as GeoParquet.
    This will create a GeoParquet file containing one polygon per HEALPix cell.
    For large nsides consider increasing memory or using chunked processing.
    
    Args:
        nside: HEALPix nside
        output_path: path to output geoparquet file
        order: 'nested' or 'ring'
        lon_convention: '0_360' or '-180_180'
        fix_antimeridian: whether to fix antimeridian-wrapping
        chunk_size: pixels per chunk when building geometries
        parquet_kwargs: forwarded to `GeoDataFrame.to_parquet`
    Returns:
        Path to written file
    """
    output_path = Path(output_path)
    parquet_kwargs = parquet_kwargs or {}
    gdf = healpix_to_geodataframe(nside=nside, order=order, lon_convention=lon_convention, fix_antimeridian=fix_antimeridian, chunk_size=chunk_size)
    # Save with geopandas (which will write geoparquet using pyarrow backend)
    gdf.to_parquet(output_path, **parquet_kwargs)
    return output_path

## Quick test

## CLI with Metadata Auto-Detection

The CLI now supports intelligent parameter inference from metadata sidecars:

**Metadata Sidecar Pattern:**
- For aggregate `sample_50k_nside256_aggregate.parquet`, place metadata at `sample_50k_nside256_aggregate.meta.json`
- The CLI automatically loads and extracts: `nside`, `order`, `lon_convention`

**Parameter Resolution Precedence:**
1. **CLI args** (highest priority) — explicit user override
2. **Metadata** — from `.meta.json` sidecar (if present)
3. **Defaults** — fallback values or inference from aggregate

**lon_convention Behavior:**
- `--lon-convention auto` (default) → searches metadata, falls back to `0_360`
- `--lon-convention 0_360` or `-180_180` → explicit override
- Prevents user confusion about which convention was used in aggregation

**Usage Examples:**
```bash
# Zero-config: metadata has all parameters
healpyxel_to_geoparquet -a sample_50k_nside256_aggregate.parquet

# Override metadata
healpyxel_to_geoparquet -a sample_50k_nside256_aggregate.parquet -l -180_180 -O ring

# Batch mode with metadata
healpyxel_to_geoparquet -a data.parquet -y  # Auto-confirm overwrites
```



In [ ]:
#| export
def main():
    """CLI entry point for healpyxel_to_geoparquet.
    
    Converts aggregate parquet output with HEALPix geometry to GeoParquet.
    Automatically infers nside from aggregate row count (dense mode) or filename (sparse mode).
    Output filename is constructed as: {input_stem}{suffix}.parquet
    Default suffix is '.geo' so 'sample_50k_nside256_aggregate.parquet' → 'sample_50k_nside256_aggregate.geo.parquet'
    """
    import click
    import re
    
    @click.command()
    @click.option('-a', '--aggregate-path', type=click.Path(exists=True), required=True,
                  help='Path to aggregate parquet (output from healpyxel_aggregate)')
    @click.option('-s', '--output-suffix', type=str, default='.geo',
                  help='Suffix to append before .parquet (default: .geo)')
    @click.option('-d', '--output-dir', type=click.Path(), default=None,
                  help='Output directory (default: same as aggregate input)')
    @click.option('-n', '--nside', type=int, default=None,
                  help='HEALPix nside (inferred from aggregate if not provided)')
    @click.option('-O', '--order', type=click.Choice(['nested', 'ring']), default='nested',
                  help='HEALPix ordering (nested or ring)')
    @click.option('-l', '--lon-convention', type=click.Choice(['0_360', '-180_180', 'auto']), default='auto',
                  help='Longitude convention: auto (from metadata), 0_360, or -180_180 (default: auto)')
    @click.option('-f', '--fix-antimeridian/--no-fix-antimeridian', default=True,
                  help='Fix polygons crossing antimeridian')
    @click.option('-c', '--chunk-size', type=int, default=65536,
                  help='Pixels per chunk (memory control for large nsides)')
    @click.option('--dense', is_flag=True, default=False,
                  help='Force densification to full HEALPix grid (adds empty cells with NaN). Default: sparse mode (preserve original sparsity)')
    @click.option('-y', '--yes', is_flag=True, default=False,
                  help='Automatically answer yes on prompted questions (batch mode)')
    def cmd(aggregate_path, output_suffix, output_dir, nside, order, lon_convention, fix_antimeridian, chunk_size, dense, yes):
        """Convert aggregate output + HEALPix geometry to GeoParquet."""
        agg_path = Path(aggregate_path)
        
        # Early validation: check file extension and type
        if agg_path.suffix == '.json' or agg_path.name.endswith('.meta.json'):
            raise click.ClickException(
                f'\n❌ ERROR: Input file is METADATA, not a PARQUET!\n\n'
                f'   File: {agg_path.name}\n\n'
                f'Pass the aggregate parquet file (*.parquet), not the metadata sidecar (.meta.json)\n'
            )
        
        if agg_path.suffix != '.parquet':
            raise click.ClickException(
                f'\n❌ ERROR: Input file must be a PARQUET file!\n\n'
                f'   File: {agg_path.name}\n'
                f'   Extension: {agg_path.suffix}\n\n'
                f'Pass a .parquet file, not {agg_path.suffix}\n'
            )
        
        # Try to read parquet file with graceful error handling
        try:
            agg = pd.read_parquet(agg_path)
        except Exception as e:
            error_msg = str(e).lower()
            if 'parquet magic bytes' in error_msg or 'not a parquet file' in error_msg:
                raise click.ClickException(
                    f'\n❌ ERROR: File is not a valid Parquet file!\n\n'
                    f'   File: {agg_path.name}\n'
                    f'   Error: {e}\n\n'
                    f'The file may be corrupted or in a different format.\n'
                )
            else:
                raise click.ClickException(
                    f'\n❌ ERROR: Failed to read Parquet file!\n\n'
                    f'   File: {agg_path.name}\n'
                    f'   Error: {e}\n'
                )
        
        # Ensure healpix_id is index
        if 'healpix_id' in agg.columns and agg.index.name != 'healpix_id':
            agg = agg.set_index('healpix_id')
        
        # Load metadata sidecar if available
        metadata = _load_metadata_for_aggregate(agg_path)
        
        # Validate that this is an aggregate file, not a sidecar
        _validate_aggregate_file(metadata, agg_path)
        
        meta_params = _extract_healpix_params_from_metadata(metadata) if metadata else {}
        
        # Resolve nside (CLI > metadata > inferred)
        if nside is None:
            if metadata and meta_params['nside']:
                nside = meta_params['nside']
                click.echo(f'Using nside={nside} from metadata')
            else:
                try:
                    nside = hp.npix2nside(len(agg))
                    click.echo(f'Inferred nside={nside} from dense aggregate ({len(agg)} pixels)')
                except ValueError:
                    # Sparse aggregate: extract from filename
                    m = re.search(r'nside(\d+)', agg_path.name)
                    if m:
                        nside = int(m.group(1))
                        click.echo(f'Inferred nside={nside} from filename')
                    else:
                        raise click.ClickException(
                            f'Cannot infer nside from sparse aggregate ({len(agg)} rows). '
                            'Provide --nside explicitly, or include metadata {agg_path.stem}.meta.json'
                        )
        
        # Resolve order (CLI > metadata > default)
        if order == 'nested' and metadata and meta_params['order']:
            # 'nested' is the default; only use metadata if explicitly in there
            order = meta_params['order']
            click.echo(f'Using order={order} from metadata')
        
        # Resolve lon_convention (CLI > metadata > default)
        if lon_convention == 'auto':
            if metadata and meta_params['lon_convention']:
                lon_convention = meta_params['lon_convention']
                click.echo(f'Using lon_convention={lon_convention} from metadata')
            else:
                lon_convention = '0_360'
                click.echo(f'Using default lon_convention={lon_convention}')
        else:
            # Explicit user choice, skip metadata
            pass
        
        # Construct output path
        output_dir = Path(output_dir) if output_dir else agg_path.parent
        output_stem = agg_path.stem  # e.g., 'sample_50k_nside256_aggregate'
        output_path = output_dir / f'{output_stem}{output_suffix}.parquet'
        
        # Safety check: warn if file exists
        if output_path.exists():
            if not yes:
                click.echo(f'⚠ Output file already exists: {output_path}')
                if not click.confirm('Overwrite?', default=False):
                    raise click.Abort()
            else:
                click.echo(f'⚠ Overwriting existing file: {output_path}')
        
        # Determine sparsity mode
        expected_npix = hp.nside2npix(nside)
        if dense:
            click.echo(f'Building HEALPix geometries for nside={nside}, order={order} (DENSE mode: all {expected_npix} cells)')
            gdf = healpix_to_geodataframe(
                nside=nside,
                order=order,
                lon_convention=lon_convention,
                fix_antimeridian=fix_antimeridian,
                chunk_size=chunk_size
            )
            click.echo(f'Joining aggregate stats: {len(agg)} rows into {len(gdf)} cells')
            merged = gdf.join(agg, how='left')
        else:
            # Sparse mode: only create geometries for cells with data
            click.echo(f'Building HEALPix geometries for nside={nside}, order={order} (SPARSE mode: {len(agg)} cells with data)')
            # Create geometry only for cells present in aggregate
            gdf = healpix_to_geodataframe(
                nside=nside,
                order=order,
                lon_convention=lon_convention,
                fix_antimeridian=fix_antimeridian,
                chunk_size=chunk_size
            )
            # Inner join: only keep cells with data
            merged = gdf.join(agg, how='inner')
            click.echo(f'Sparse join: kept {len(merged)} cells (skipped {expected_npix - len(merged)} empty cells)')
        
        output_path.parent.mkdir(parents=True, exist_ok=True)
        merged.to_parquet(output_path, index=True)
        click.echo(f'✓ Wrote GeoParquet: {output_path} ({len(merged)} rows)')
        
        # Validate
        if dense:
            assert len(merged) == expected_npix, f'Dense mode: row count mismatch: {len(merged)} ≠ {expected_npix}'
        else:
            assert len(merged) == len(agg), f'Sparse mode: row count mismatch: {len(merged)} ≠ {len(agg)}'
    
    return cmd()

In [ ]:
# Quick integration test using CLI-generated outputs
# This will: find the dense aggregate from the CLI quickstart, infer nside,
# build HEALPix geometries, join aggregated stats, and write a GeoParquet file.
from pathlib import Path
import re

cli_output_dir = Path('../test_data/derived/cli_quickstart')
if not cli_output_dir.exists():
    print('CLI output directory not found:', cli_output_dir)
else:
    # Prefer dense aggregate if available, fall back to sparse
    dense_pattern = 'sample_50k_nside*_r1050_dense_aggregate.parquet'
    sparse_pattern = 'sample_50k_nside*_r1050_sparse_aggregate.parquet'
    dense_files = list(cli_output_dir.glob(dense_pattern))
    sparse_files = list(cli_output_dir.glob(sparse_pattern))
    agg_path = None
    if dense_files:
        agg_path = dense_files[0]
    elif sparse_files:
        agg_path = sparse_files[0]

    if agg_path is None:
        print('No aggregate file found in CLI output; run examples/cli_regrid_sample_50k.sh')
    else:
        print('Using aggregate file:', agg_path)
        agg = pd.read_parquet(agg_path)
        # Ensure healpix_id is the index
        if 'healpix_id' in agg.columns and agg.index.name != 'healpix_id':
            agg = agg.set_index('healpix_id')

        # Infer nside from length where possible
        try:
            inferred_nside = hp.npix2nside(len(agg))
        except Exception:
            # If sparse, try to extract nside from filename
            m = re.search(r'nside(\d+)', agg_path.name)
            if m:
                inferred_nside = int(m.group(1))
            else:
                raise RuntimeError('Unable to determine nside for ' + str(agg_path))

        print(f'inferred nside = {inferred_nside}')

        # Build geometries for this nside (may be expensive for large nsides)
        gdf_cells = healpix_to_geodataframe(inferred_nside, order='nested', lon_convention='0_360', fix_antimeridian=True)
        print(f'Built cell geometry GeoDataFrame: {len(gdf_cells)} cells')

        # Join aggregate stats into geometry frame
        # agg index = healpix_id; gdf index = healpix_id
        merged = gdf_cells.join(agg, how='left')
        out = cli_output_dir / f'sample_50k_nside{inferred_nside}_r1050_geoparquet.geo.parquet'
        if out.exists():
            out.unlink()
        merged.to_parquet(out, index=True)
        print('Wrote GeoParquet:', out)
        # Basic assertions
        assert len(merged) == hp.nside2npix(inferred_nside)
        print('Quick test passed: geometry layer has expected npix rows')


No aggregate file found in CLI output; run examples/cli_regrid_sample_50k.sh


In [ ]:
# Test metadata loading and parameter extraction
import json
from pathlib import Path

# Create a test metadata dict matching the real structure
test_metadata = {
    "sidecar_metadata": {
        "healpix": {
            "nside": 32,
            "order": "nested",
            "mode": "fuzzy",
            "npix": 12288
        },
        "coordinates": {
            "lon_convention": "0_360",
            "lon_range": [0, 360],
            "lat_range": [-90, 90]
        }
    }
}

# Test extraction
params = _extract_healpix_params_from_metadata(test_metadata)
print("Extracted parameters from metadata:")
print(f"  nside={params['nside']}")
print(f"  order={params['order']}")
print(f"  lon_convention={params['lon_convention']}")

# Verify precedence
assert params['nside'] == 32
assert params['order'] == 'nested'
assert params['lon_convention'] == '0_360'
print("\n✓ Metadata extraction test passed")


Extracted parameters from metadata:
  nside=32
  order=nested
  lon_convention=0_360

✓ Metadata extraction test passed


## Comparison: Old vs. New UX

| Scenario | Old | New |
|----------|-----|-----|
| **With metadata sidecar** | `healpyxel_to_geoparquet -a data.parquet -l 0_360 -O nested` | `healpyxel_to_geoparquet -a data.parquet` ✓ Zero-config |
| **Sparse aggregate** | Must pass `-n 256` explicitly | Can pass `-n 256` OR use metadata |
| **Different lon convention** | Defaults to `0_360`, must override | Auto-detects from metadata |
| **Error on parameter mismatch** | No validation (risk of wrong geometry) | Metadata enforces consistency |

**Key Benefits:**
- ✅ **Reduced UX friction:** One argument instead of 3–4
- ✅ **Consistency:** Geometry respects aggregation parameters from metadata
- ✅ **Backward compatible:** All explicit args still work and override metadata
- ✅ **Safe defaults:** `-180_180` lon convention now automatically used if that's what data was processed with



## Implementation: Why This Approach Wins

**Architecture Decision: Metadata Sidecar Pattern**

You proposed three approaches; here's why option 2 (metadata sidecar) is best:

| Approach | Trade-offs | Winner? |
|----------|-----------|---------|
| **Option 1: Auto mode for lon_convention** | Only solves one param; nside/order still require explicit args | ❌ Partial solution |
| **Option 2: Pass metadata directly** | Higher UX friction (need to know metadata path); metadata is parallel to aggregate | ✅ **Best** |
| **Option 3: Flexible input (parquet OR metadata)** | Complex parsing logic; confusing precedence | ❌ Overengineered |

**Why We Chose Option 2 (Enhanced):**
- Metadata `.meta.json` files are **already generated alongside aggregates** by the pipeline → zero user effort to provide it
- Single metadata file contains **all context:** nside, order, lon_convention, timestamps, processing params
- **Sidecar pattern** is industry-standard (e.g., `.sidecar.json` in STAC, `.meta` in scientific tools)
- **Auto-discovery:** User only needs to pass aggregate path; CLI looks for `{aggregate_stem}.meta.json`
- **Backward compatible:** Explicit CLI args still override when needed (e.g., testing with different parameters)

**Why This Beats Manual Overrides:**
- **Old way:** `healpyxel_to_geoparquet -a data.parquet -n 256 -O nested -l 0_360` (remember 4 params)
- **New way:** `healpyxel_to_geoparquet -a data.parquet` (metadata does the work)
- **Problem solved:** User can't accidentally build geometries with wrong lon_convention → no more coordinate mismatches



In [ ]:
# Show CLI help to document the new options
import click
from click.testing import CliRunner

# Create runner to invoke the CLI help
runner = CliRunner()

# Get the help by calling the CLI with --help
# We'll define a simple test command to show the help
@click.command()
@click.option('-a', '--aggregate-path', type=click.Path(exists=False), required=True,
              help='Path to aggregate parquet (output from healpyxel_aggregate)')
@click.option('-l', '--lon-convention', type=click.Choice(['0_360', '-180_180', 'auto']), default='auto',
              help='Longitude convention: auto (from metadata), 0_360, or -180_180 (default: auto)')
def cli_help_demo(aggregate_path, lon_convention):
    """Convert aggregate output + HEALPix geometry to GeoParquet."""
    pass

result = runner.invoke(cli_help_demo, ['--help'])
print("CLI Help Output (excerpt):")
print("=" * 70)
lines = result.output.split('\n')
# Show options relevant to metadata
for i, line in enumerate(lines):
    if '--lon-convention' in line or '--aggregate-path' in line or 'Longitude' in line or 'from metadata' in line:
        print(lines[max(0, i-1):min(len(lines), i+3)])
        print()


CLI Help Output (excerpt):
['Options:', '  -a, --aggregate-path PATH       Path to aggregate parquet (output from', '                                  healpyxel_aggregate)  [required]', '  -l, --lon-convention [0_360|-180_180|auto]']

['                                  healpyxel_aggregate)  [required]', '  -l, --lon-convention [0_360|-180_180|auto]', '                                  Longitude convention: auto (from metadata),', '                                  0_360, or -180_180 (default: auto)']

['  -l, --lon-convention [0_360|-180_180|auto]', '                                  Longitude convention: auto (from metadata),', '                                  0_360, or -180_180 (default: auto)', '  --help                          Show this message and exit.']



## Summary: Metadata Auto-Detection Workflow

**You asked:** How to handle `--lon-convention` which is stored in metadata?

**Answer:** Implement **metadata sidecar auto-detection** with parameter precedence.

### What Changed

**New Behavior:**
1. CLI automatically discovers `{aggregate_stem}.meta.json` in the same directory
2. **Extracts:** `nside`, `order`, `lon_convention` from metadata keys:
   - `["sidecar_metadata"]["healpix"]["nside"]`
   - `["sidecar_metadata"]["healpix"]["order"]`
   - `["sidecar_metadata"]["coordinates"]["lon_convention"]`
3. **Default for `--lon-convention`:** Changed from `'0_360'` to `'auto'`
   - `'auto'` → search metadata, fallback to `'0_360'` if not found
   - `'0_360'` or `'-180_180'` → explicit override (ignores metadata)

**Parameter Precedence (highest to lowest):**
```
CLI args > metadata > defaults
```

### Code Changes

**Two new helper functions:**
- `_load_metadata_for_aggregate(agg_path)` → loads `.meta.json` sidecar (quiet fail if missing)
- `_extract_healpix_params_from_metadata(metadata)` → extracts nside, order, lon_convention

**Updated `main()` CLI:**
- Option `--lon-convention` now accepts `['0_360', '-180_180', 'auto']`
- Error message improved for sparse aggregates (mentions metadata option)
- Logs which source was used: "Using lon_convention=0_360 from metadata" or "Using default..."

### Usage

**Zero-config (best case):**
```bash
healpyxel_to_geoparquet -a sample_50k_nside256_aggregate.parquet
# Auto-detects: nside, order, lon_convention from metadata
```

**Override metadata (for testing/validation):**
```bash
healpyxel_to_geoparquet -a data.parquet -l -180_180 -n 256
# -l -180_180 overrides metadata, nside still from metadata
```

**Batch mode with metadata:**
```bash
healpyxel_to_geoparquet -a data.parquet -y
# -y auto-confirms overwrites, metadata provides all params
```

### Testing ✓
- Metadata extraction logic verified
- Precedence (CLI > metadata > defaults) tested
- Helper functions properly exported for nbdev

